# DNA tático — Barcelona vs Real Madrid (La Liga 2015/2016)

Dois clásicos: **266424** (Bernabéu) e **267533** (Camp Nou), via StatsBomb Open Data.

## Imports e carregamento dos dois Clásicos

Os eventos vêm de `lib_analise.load_all()`, que concatena `events/266424.json` e `events/267533.json`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Importa lib_analise com o kernel na raiz do repositório ou em analise/
_here = Path.cwd()
_analise = _here / "analise" if (_here / "analise" / "lib_analise.py").exists() else _here
if str(_analise) not in sys.path:
    sys.path.insert(0, str(_analise))

from lib_analise import MATCHES, TEAMS, load_all, load_events

df = load_all()
print("Jogos carregados:")
for match_id, meta in MATCHES.items():
    n = (df["match_id"] == match_id).sum()
    print(f"  {match_id}: {meta['label']} — {n} eventos")
print(f"\nTotal: {len(df)} eventos | Times: {TEAMS}")
df.head()

Jogos carregados:
  266424: Bernabeu (21/11/2015) — 3860 eventos
  267533: Camp Nou (02/04/2016) — 3700 eventos

Total: 7560 eventos | Times: ['Barcelona', 'Real Madrid']


,match_id,id,index,period,minute,second,type,team,player,position,...,y,pass_end_x,pass_end_y,pass_length,pass_height,pass_cross,pass_outcome,pass_recipient,shot_xg,shot_outcome
0,266424,5affd831-590f-4e98-a908-d08bd4e5b869,1,1,0,0,Starting XI,Real Madrid,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
1,266424,ec49ca6b-4eb5-4d10-8e2c-981a33b5e970,2,1,0,0,Starting XI,Barcelona,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
2,266424,6d900b31-be0e-4a53-b058-38c98d4dc9da,3,1,0,0,Half Start,Real Madrid,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
3,266424,2d93744e-e07b-4d65-8353-a7321e2124bb,4,1,0,0,Half Start,Barcelona,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
4,266424,81777385-a4cd-47df-ac69-950ea2965b74,5,1,0,0,Pass,Barcelona,Neymar da Silva Santos Junior,Left Wing,...,40.1,60.3,43.8,3.765634,Ground Pass,False,NaN,Luis Alberto Suárez Díaz,NaN,NaN


## Métricas M1–M6 (DNA tático)

Cada função em `lib_analise` agrega os **dois jogos** por time, conforme o plano: construção (M1), progressão (M2), lateralização no terço de ataque (M3), entradas na área e cruzamentos (M4), altura de recuperação (M5) e ameaça (M6).

In [2]:
from IPython.display import display

from lib_analise import (
    build_dna_table,
    export_dna_table_csv,
    metric_m1,
    metric_m2,
    metric_m3,
    metric_m4,
    metric_m5,
    metric_m6,
)

m1 = metric_m1(df)
m2 = metric_m2(df)
m3 = metric_m3(df)
m4 = metric_m4(df)
m5 = metric_m5(df)
m6 = metric_m6(df)

display(m1.round(4))
display(m2.round(4))
display(m3.round(4))
display(m4.round(4))
display(m5.round(4))
display(m6.round(4))

dna = build_dna_table(df)
csv_path = export_dna_table_csv(df)
display(dna.round(2))
print(f"Tabela DNA exportada: {csv_path.resolve()}")

,posses_total,media_passes_por_posse,mediana_passes_por_posse,posses_longas_5plus,duracao_media_min_posse,posses_por_jogo_media
team,,,,,,
Barcelona,187,7.4492,4.0,0.4973,0.4011,93.5
Real Madrid,189,4.3704,3.0,0.3651,0.2487,94.5


,avanco_medio_m,pct_progressivos,passes_completos
team,,,
Barcelona,1.8257,0.1982,1236
Real Madrid,4.3533,0.2976,662


lado,direito,esquerdo,total,pct_esq,pct_dir
team,,,,,
Barcelona,550,363,913,0.3976,0.6024
Real Madrid,306,368,674,0.5460,0.4540


,cruzamentos,total_passes_para_area,passes_centro_ou_pela_area,pct_cruzamentos
team,,,,
Barcelona,7,44,37,0.1591
Real Madrid,7,45,38,0.1556


,x_medio_recuperacao,n_eventos_defensivos,pressure_count
team,,,
Barcelona,54.5988,341,194
Real Madrid,51.5586,464,315


,chutes_por_jogo,xg_por_jogo,gols_total
team,,,
Barcelona,16.0,1.6022,5
Real Madrid,14.0,1.3318,2


## Tabela DNA consolidada

A função `build_dna_table` em `lib_analise` junta as métricas M1–M6. Exportamos CSV em `figuras/` (na raiz do repositório).

In [ ]:
from pathlib import Path

from lib_analise import build_dna_table

dna = build_dna_table(df)
_repo = Path.cwd().resolve()
_fig = _repo / "figuras" if (_repo / "figuras").exists() else _repo.parent / "figuras"
_fig.mkdir(parents=True, exist_ok=True)
dna.round(4).to_csv(_fig / "tabela_dna.csv")
dna

## Visualizações (geração reprodutível)

O script abaixo grava todos os PNGs em `figuras/` (mesma lógica que `python analise/run_gerar_figuras.py` executado na raiz do projeto).

In [ ]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    p = Path.cwd().resolve()
    if (p / "analise" / "run_gerar_figuras.py").exists():
        return p
    if (p.parent / "analise" / "run_gerar_figuras.py").exists():
        return p.parent
    raise FileNotFoundError("Abra o notebook a partir de trabalho-ga/ ou trabalho-ga/analise/")

root = _repo_root()
subprocess.check_call([sys.executable, str(root / "analise" / "run_gerar_figuras.py")])
print("Figuras em:", root / "figuras")